In [1]:
import importlib

import numpy as np
import pandas as pd 
import geopandas as gpd
from pathlib import Path

from tqdm.auto import tqdm
tqdm.pandas()

import results_utils
importlib.reload(results_utils)

from results_utils import load_bootstrap_eco, load_bootstrap_pa

output_path = '/marbec-data/RLS-Australia/malpolon/outputs/'

## Eco indic

In [82]:
td_eco_preds = '43_td_eco_preds-2026-03-30_16-35'
td_cp_eco = '42_td_eco_seedtest-2026-03-10_14-13'

inputpath = Path('/marbec-data/RLS-Australia/malpolon/inputs/australiapreds')
preds_mean, preds_ci = load_bootstrap_eco(output_path, td_eco_preds, td_cp_eco)
metadata = pd.read_csv(inputpath / 'auspredgrid.csv', index_col='survey_id', usecols=['survey_id', 'depth', 'latitude', 'longitude', 'eventDate'])

In [ ]:
ecopreds= pd.merge(left=metadata, right=preds_mean, left_index=True, right_index=True)
ecoci = pd.merge(left=metadata, right=preds_ci, left_index=True, right_index=True)
dates = ecopreds['eventDate'].drop_duplicates().sort_values()

for var in ['log_sr', 'log_uicn_sr']:
    for d in list(dates):
        # results_utils.export_map( predictions_mean=ecopreds[ecopreds['eventDate'] == d],
        #             predictions_ci=ecoci[ecoci['eventDate'] == d],
        #             var_name=var,
        #             out_path=f"/home/gaetan/maps/{var}",
        #             filename=f"{var}_{d}")
        results_utils.convert_to_png(f"/home/gaetan/maps/{var}", f"{var}_{d}.tif", colormap = 'turbo',
                                     uncertainty_threshold=0.2, relative_threshold=True, hatch_color='black', hatch_size=30)

## Calculate % of cells

In [2]:
td_eco_preds = '43_td_eco_preds-2026-03-30_16-35'
td_cp_eco = '42_td_eco_seedtest-2026-03-10_14-13'

inputpath = Path('/marbec-data/RLS-Australia/malpolon/inputs/australiapreds')
preds_mean, preds_ci = load_bootstrap_eco(output_path, td_eco_preds, td_cp_eco)
metadata = pd.read_csv(inputpath / 'auspredgrid.csv', index_col='survey_id', usecols=['survey_id', 'depth', 'latitude', 'longitude', 'eventDate'])

metadata['log_uicn_sr'] = preds_mean.loc[metadata.index, 'log_uicn_sr']
metadata['log_uicn_sr_ci'] = preds_ci.loc[metadata.index, 'log_uicn_sr']

In [3]:
preds_gpd = gpd.GeoDataFrame(metadata, geometry=gpd.points_from_xy(metadata.longitude, metadata.latitude), crs='EPSG:4326')
cats = ['<1', '1-1.5', '1.5-2', '2-3', '>3']
preds_gpd['pred_cat'] = pd.cut(preds_gpd['log_uicn_sr'], bins=[0, 1, 1.5, 2, 3, np.inf], labels=cats)


mpas = gpd.read_file('/home/gaetan/Downloads/aus_highly_protected.gpkg')
preds_gpd["geometry"] = preds_gpd["geometry"].buffer(0.05, cap_style="square")
preds_gpd['in_mpa'] = preds_gpd.geometry.apply(lambda x: mpas.intersects(x).any())

/tmp/ipykernel_100408/4010046414.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  preds_gpd["geometry"] = preds_gpd["geometry"].buffer(0.05, cap_style="square")


In [4]:
summary = pd.DataFrame()
summary['inside'] = preds_gpd.loc[preds_gpd['in_mpa'] == True, 'pred_cat'].value_counts()
summary['outside'] = preds_gpd.loc[preds_gpd['in_mpa'] == False, 'pred_cat'].value_counts()
summary.loc['Any'] = [len(preds_gpd[preds_gpd['in_mpa'] == True]), len(preds_gpd[preds_gpd['in_mpa'] == False])]
summary = summary.reindex(['Any'] + cats)
summary = summary / len(preds_gpd)
summary['percent'] = summary['inside'] / (summary['outside'] + summary['inside'])

In [5]:
from plotly import graph_objects as go
import plotly.express as px

fig = go.Figure()

colors = px.colors.qualitative.Pastel

fig.add_trace(go.Bar(
    name='Cells outside highly protected areas',
    x=summary.index, y=summary['outside'],
    marker_color=colors[1]
))

fig.add_trace(go.Bar(
    name='Cells partly inside highly protected areas',
    x=summary.index, y=summary['inside'],
    marker_color=colors[0],
    text=summary['percent'], 
    texttemplate='%{text:.0%}',   
    textposition='outside',   
    textfont=dict(color=colors[0])
))



fig.update_layout(barmode='stack',
                  width = 800, height=800,
                  template = 'simple_white',
                  title='',
                  xaxis_title='Predicted Threatened Species Richness',
                  yaxis_title='Proportion of all cells',
                  font=dict(size=20),
                  legend_title_text='',
                  legend=dict(yanchor="top",
                              xanchor="right"),
)

fig.show()

## Blind spots

In [2]:
td_eco_preds = '43_td_eco_preds-2026-03-30_16-35'
td_cp_eco = '42_td_eco_seedtest-2026-03-10_14-13'

inputpath = Path('/marbec-data/RLS-Australia/malpolon/inputs/australiapreds')
preds_mean, preds_ci = load_bootstrap_eco(output_path, td_eco_preds, td_cp_eco)
metadata = pd.read_csv(inputpath / 'auspredgrid.csv', index_col='survey_id', usecols=['survey_id', 'depth', 'latitude', 'longitude', 'eventDate'])

In [ ]:
ecopreds= pd.merge(left=metadata, right=preds_mean, left_index=True, right_index=True)
ecoci = pd.merge(left=metadata, right=preds_ci, left_index=True, right_index=True)
dates = ecopreds['eventDate'].drop_duplicates().sort_values()

var = 'log_uicn_sr'
d = list(dates)[0]
results_utils.blindspots_map(f"/home/gaetan/maps/{var}", f"{var}_{d}.tif", crop={'left': 0.2, 'right': 0.8, 'bottom': 0, 'top': 1}, cmap='RdYlGn_r')

## UICN P-A

In [42]:
td_pa_preds = '43_td_pa_preds-2026-03-31_17-02'

species = ['Parascyllium variolatum']#, 'Hypoplectrodes maccullochi', 'Parma unifasciata']

inputpath = Path('/marbec-data/RLS-Australia/malpolon/inputs/australiapreds')
preds_mean, preds_ci = load_bootstrap_pa(output_path, td_pa_preds, usecols = ['survey_id'] + species)
metadata = pd.read_csv(inputpath / 'auspredgrid.csv', index_col='survey_id', usecols=['survey_id', 'depth', 'latitude', 'longitude', 'eventDate'])

In [ ]:
papreds= pd.merge(left=metadata, right=preds_mean, left_index=True, right_index=True)
paci = pd.merge(left=metadata, right=preds_ci, left_index=True, right_index=True)
dates = papreds['eventDate'].drop_duplicates().sort_values()

for s in species:
    Path(f"/home/gaetan/maps/{s}").mkdir(parents=True, exist_ok=True)
    for d in list(dates)[:1]:
        # results_utils.export_map( predictions_mean=papreds[papreds['eventDate'] == d],
        #             predictions_ci=paci[paci['eventDate'] == d],
        #             var_name=s,
        #             out_path=f"/home/gaetan/maps/{s}",
        #             filename=f"{s}_{d}")
        results_utils.convert_to_png(f"/home/gaetan/maps/{s}", f"{s}_{d}.tif", colormap = 'GnBu',
                                     species = True, uncertainty_threshold=0.005, relative_threshold=False, hatch_color='red', hatch_size = 50,
                                    crop = {'left':0.12, 'right':0.8, 'bottom':0, 'top':0.5}, title_coords = (0.2, 0.3))